# Tutorial 2: How can linear regression model a curved relationship?

## Briefing for the responsible study group

| Planning item | Allocation |
|---|---|
| Presentation date | Friday 20 November 2026 |
| Prerequisite lecture | Forecasting & Prediction, 10 November 2026 |
| Preparation time after prerequisite | 7 full weekdays, excluding lecture and tutorial dates |
| Format | 15-minute presentation + approximately 15-minute discussion |

**Lecture connection:** Forecasting slides 12–20 and 29–40: linear models, model fitting, held-out evaluation and cross-validation.

This is your group's working notebook. It replaces the requirement to augment a section of an existing course notebook. The supplied cells establish a starting point; your group develops and teaches the investigation.

### Your teaching challenge

Many students interpret “linear regression” as “a straight line in the original feature.” Show why a regression can remain linear in its parameters while modelling a curved relationship in age. Then investigate why extra flexibility can help up to a point and become counterproductive.

Use the supplied synthetic relationship between an item's age and resale price. Compare polynomial degrees **1, 2 and 8**, using the same small training sample and validation set. The true relationship is supplied so that you can distinguish systematic structure from noise; a real analyst would not know it.

### Core investigation

1. Explain the difference between being linear in the coefficients and being linear in the original feature. Write the degree-2 prediction function and identify its fitted parameters.
2. Fit the three candidate models. Plot fitted curves against the observations, and compare training and validation RMSE.
3. Investigate sensitivity to the particular training sample using a small number of repeats, keeping the validation and test observations fixed across repeats. Do not draw a general conclusion from one lucky or unlucky fit. Choose a degree using validation evidence; use the test set once for the final report after this choice.
4. Explain why a curve that follows training observations closely can be less useful for future predictions. Describe the roles of sample size, noise and flexibility without requiring a formal bias–variance decomposition.

### What classmates should gain

They should understand feature engineering as a way to change the relationships a simple model can represent, and why model choice must consider unseen observations. Do not promise that degree 8 always loses or degree 2 always wins: show what your experiment supports.

### Boundaries and support

Use `PolynomialFeatures` and `LinearRegression` in a pipeline. There is no need to derive normal equations, search many degrees or introduce ridge/lasso regression. Normalized ages are supplied to reduce numerical scaling difficulties. Keep the main evaluation within the observed age range. **Optional:** illustrate extrapolation separately and explain why it answers a different question.

## How to turn your investigation into a tutorial

Your group has **15 minutes to present**, followed by approximately **15 minutes of discussion** with the instructor and class. Your audience has attended the prerequisite lecture but has not done your investigation. Help them understand one important question through a focused, reproducible demonstration.

**Required contribution.** Complete the core investigation below, explain why the results occur, and discuss at least one limitation or contrasting condition. Select two or three main figures or tables that make your argument clear. A useful tutorial connects the question, experimental design, evidence and conclusion; a sequence of unexplained code cells is insufficient. You are not expected to build a production system, implement algorithms from scratch, collect external data or run an extensive model search.

**What to submit.** Submit your completed `.ipynb` notebook with explanations and reproducible code, plus any slides or other presentation material you use. Follow the instructor's announced submission deadline. Keep the supplied briefing, but edit, reorganize or replace the working sections as needed. Identify substantial changes to supplied helpers. Record package versions, seeds and relevant settings; restart the kernel and run all cells before submission. Clearly distinguish supplied material from your group's contribution.

**Presentation format is your choice.** You may present from the notebook, create slides, render to HTML, develop an interactive demonstration or combine these. Make the selected format work within 15 minutes and have a fallback for live demonstrations. Creativity earns credit when it improves understanding, engagement or the quality of the investigation. Technical spectacle should be avoided. A clear static plot can be excellent.

**AI assistance is allowed.** You may use AI to help develop code, explanations and visualizations. Your group remains responsible for correctness and must be able to explain the submitted work. Include a short account of the tools used, their main contributions, and how you checked those contributions. Verify at least one important result independently—for example, with a hand calculation, an analytic special case or a comparison with a trusted library function. A long prompt transcript is not required. Using AI is optional and is not itself rewarded.

**Assessment.** This component contributes **15% of the course grade**. Within the tutorial, the criteria are **conceptual understanding (40%)**, **added value for classmates (40%)**, and **quality of communication (20%)**. Added value comes from the investigation and how it develops understanding beyond the lecture and the supplied starter cells. Optional extensions are not required for an excellent grade. All groups should prioritize a sound core investigation over breadth.

**Suggested presentation rhythm:** about 3 minutes for the question and essential concepts, 8 minutes for the demonstration and evidence, and 4 minutes for interpretation and limitations. Prepare one question that invites the class to reason about your findings. Anticipate questions about experimental controls, assumptions and alternative explanations.

## Preliminaries: environment

The course environment provides all libraries needed to execute the starter code (see the [README for installation instructions](https://github.com/stefanlessmann/ESMT_IML)). You can install additional libraries if needed.

Run the supplied cells first. They provide a starting point, **not a completed tutorial**. Code cells marked `YOUR GROUP'S WORK` are intentionally unfinished and can be replaced or expanded.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 888
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.size': 11})
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')

## Preliminaries: supplied data and helpers

The training set is intentionally small. Validation and test observations are generated independently from the same age range. Generate the initial splits once with `make_polynomial_data()`. For repeats, generate only a new training sample with `sample_resale(28, np.random.default_rng(training_seed))`, changing `training_seed` and retaining the original validation and test arrays. Calling `make_polynomial_data` with a different seed would change all three splits. Keep the test set aside until the final model choice. `true_price` is a teaching reference only; do not use it as an input feature or an extra training target.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def true_price(age):
    z = np.asarray(age) / 48.0
    return 1050 - 1100 * z + 400 * z**2

def sample_resale(n, rng, noise=65.0):
    age = rng.uniform(3, 48, n)
    return age[:, None] / 48.0, true_price(age) + rng.normal(0, noise, n)

def make_polynomial_data(seed=SEED):
    rng = np.random.default_rng(seed)
    X_train, y_train = sample_resale(28, rng)
    X_val, y_val = sample_resale(200, rng)
    X_test, y_test = sample_resale(400, rng)
    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = make_polynomial_data()
degrees = [1, 2, 8]
age_grid = np.linspace(3, 48, 300)
X_grid = age_grid[:, None] / 48.0
display(pd.DataFrame({'age_months': X_train[:, 0] * 48,
                      'resale_price': y_train}).head())

## Investigation 1: Build and explain the three models

Use a separate pipeline for each degree, with PolynomialFeatures(include_bias=False) and LinearRegression(). Fit only on training data. Explain the transformed features.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 2: Compare flexibility and generalization

Plot the three curves and report training/validation RMSE. Repeat with a few training samples using `sample_resale` as described above; hold validation and test observations fixed. Within each repeat, all degrees must use the same training observations. Separate the effects of changing the training sample from changing model degree.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 3: Make and defend a choice

Choose a degree using validation results, explain your selection rule, and then report test error. Include a caveat and a short independent check of a polynomial prediction.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Your teaching conclusions

Replace this text with:

1. A direct answer to the tutorial's central question, supported by your results.
2. Two or three takeaways that classmates should remember.
3. One limitation, counterexample or condition under which the result changes.
4. One discussion question, plus notes on how your group would answer it.

## Contribution and verification record

- **Supplied material retained or changed:** …
- **Your group's experimental and explanatory contributions:** …
- **Independent check of an important result:** …
- **AI tools and their contributions, if any:** …
- **How generated code/explanations were checked:** …
- **Package versions, random seeds and reproduction instructions:** …
- **Group members and contributions:** …

## Before presenting

- [ ] Restarted the kernel and ran all cells successfully.
- [ ] Checked that splits, metrics, units and class labels are correct.
- [ ] Distinguished evidence from interpretation; avoided claims stronger than the experiment supports.
- [ ] Labelled plots and made figures legible for classmates.
- [ ] Rehearsed within 15 minutes and prepared for discussion.
- [ ] Included the completed notebook and any additional presentation material.